In [1]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import pandas as pd

# 加载英文情绪分类模型
model_path = "my_final_model/E_model"
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

print("✅ English Emotion Model Loaded Successfully!")

# 英文情绪映射
emotion_mapping = {
    0: "sadness",
    1: "joy", 
    2: "love",
    3: "anger",
    4: "fear",
    5: "surprise"
}

# 情绪表情符号
emotion_emojis = {
    "sadness": "😢",
    "joy": "😊", 
    "love": "❤️",
    "anger": "😠",
    "fear": "😨",
    "surprise": "😲"
}

def predict_emotion_english(text):
    """预测英文文本的情绪"""
    # 编码文本
    inputs = tokenizer(
        text, 
        return_tensors="pt", 
        truncation=True, 
        padding=True, 
        max_length=128
    )
    
    # 模型预测
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
    
    # 获取所有情绪的概率
    probabilities = predictions[0].numpy()
    
    # 找到主要情绪
    predicted_class = probabilities.argmax()
    confidence = probabilities[predicted_class]
    emotion = emotion_mapping[predicted_class]
    emoji = emotion_emojis.get(emotion, "❓")
    
    # 获取所有情绪的概率（按置信度排序）
    emotion_probs = []
    for i, prob in enumerate(probabilities):
        emotion_name = emotion_mapping[i]
        emotion_emoji = emotion_emojis.get(emotion_name, "❓")
        emotion_probs.append({
            'emotion': emotion_name,
            'emoji': emotion_emoji,
            'confidence': prob
        })
    
    emotion_probs.sort(key=lambda x: x['confidence'], reverse=True)
    
    return {
        'text': text,
        'primary_emotion': emotion,
        'primary_emoji': emoji,
        'primary_confidence': confidence,
        'all_emotions': emotion_probs
    }

# 测试各种英文情绪
test_texts = [
    # Sadness
    "I feel so lonely and empty inside, like there's a void that can never be filled.",
    "The rain outside matches the storm in my heart, each drop a reminder of what I've lost.",
    
    # Joy
    "I'm overflowing with happiness! Everything feels perfect and full of possibilities.",
    "This is the best day of my life! I can't stop smiling and feeling grateful.",
    
    # Love
    "My heart swells with affection every time I think of you, you complete me.",
    "The way you care for others shows the beautiful soul you have, I admire you deeply.",
    
    # Anger
    "I'm absolutely furious! How could they be so irresponsible and thoughtless?",
    "This injustice makes my blood boil, I can't believe they would do something like this.",
    
    # Fear
    "My heart is racing and I can't breathe properly, I'm terrified of what might happen.",
    "The uncertainty of the future fills me with dread and anxiety.",
    
    # Surprise
    "Oh my god! I never expected this wonderful news, I'm completely shocked!",
    "This revelation has left me speechless, I had no idea about any of this.",
    
    # Mixed/Complex
    "I don't know whether to laugh or cry, this situation is so unexpected and emotional.",
    "Part of me is excited but another part is scared, it's such a confusing mix of feelings."
]

print("🧪 English Emotion Classification Test Results:")
print("=" * 80)

for text in test_texts:
    result = predict_emotion_english(text)
    
    print(f"📝 Text: {result['text']}")
    print(f"🎯 Primary Emotion: {result['primary_emoji']} {result['primary_emotion']} (Confidence: {result['primary_confidence']:.2%})")
    
    print("📊 Detailed Probabilities:")
    for i, emotion_info in enumerate(result['all_emotions'][:3]):  # Show top 3
        print(f"   {i+1}. {emotion_info['emoji']} {emotion_info['emotion']}: {emotion_info['confidence']:.2%}")
    
    print("-" * 80)

✅ English Emotion Model Loaded Successfully!
🧪 English Emotion Classification Test Results:
📝 Text: I feel so lonely and empty inside, like there's a void that can never be filled.
🎯 Primary Emotion: 😢 sadness (Confidence: 99.65%)
📊 Detailed Probabilities:
   1. 😢 sadness: 99.65%
   2. 😨 fear: 0.09%
   3. ❤️ love: 0.07%
--------------------------------------------------------------------------------
📝 Text: The rain outside matches the storm in my heart, each drop a reminder of what I've lost.
🎯 Primary Emotion: 😢 sadness (Confidence: 99.22%)
📊 Detailed Probabilities:
   1. 😢 sadness: 99.22%
   2. 😊 joy: 0.31%
   3. 😠 anger: 0.20%
--------------------------------------------------------------------------------
📝 Text: I'm overflowing with happiness! Everything feels perfect and full of possibilities.
🎯 Primary Emotion: 😊 joy (Confidence: 99.68%)
📊 Detailed Probabilities:
   1. 😊 joy: 99.68%
   2. ❤️ love: 0.11%
   3. 😢 sadness: 0.09%
----------------------------------------------------

In [ ]:
def interactive_english_test():
    """交互式英文情绪测试"""
    print("🎭 English Emotion Analyzer")
    print("Enter English text to analyze emotion, type 'quit' to exit")
    print("=" * 60)
    
    while True:
        text = input("\nEnter English text: ").strip()
        
        if text.lower() in ['quit', 'exit', 'q']:
            print("Goodbye! 👋")
            break
            
        if not text:
            continue
            
        # 预测情绪
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
        
        model.eval()
        with torch.no_grad():
            outputs = model(**inputs)
            predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
        
        probabilities = predictions[0].numpy()
        predicted_class = probabilities.argmax()
        confidence = probabilities[predicted_class]
        emotion = emotion_mapping[predicted_class]
        emoji = emotion_emojis.get(emotion, "❓")
        
        print(f"\n📊 Analysis Result:")
        print(f"   Text: {text}")
        print(f"   Primary Emotion: {emoji} {emotion} (Confidence: {confidence:.2%})")
        
        # 显示前3个可能的情绪
        sorted_indices = probabilities.argsort()[::-1]
        print("   Other possible emotions:")
        for i in range(1, min(4, len(sorted_indices))):
            idx = sorted_indices[i]
            other_emotion = emotion_mapping[idx]
            other_emoji = emotion_emojis.get(other_emotion, "❓")
            other_conf = probabilities[idx]
            print(f"     {other_emoji} {other_emotion}: {other_conf:.2%}")

# 运行交互式测试
interactive_english_test()

🎭 English Emotion Analyzer
Enter English text to analyze emotion, type 'quit' to exit



Enter English text:  oh I really love this fake product, really really good



📊 Analysis Result:
   Text: oh I really love this fake product, really really good
   Primary Emotion: 😢 sadness (Confidence: 98.68%)
   Other possible emotions:
     😨 fear: 0.45%
     😠 anger: 0.36%
     😊 joy: 0.31%



Enter English text:  I love this XiaoGong adult comic book



📊 Analysis Result:
   Text: I love this XiaoGong adult comic book
   Primary Emotion: 😊 joy (Confidence: 87.93%)
   Other possible emotions:
     😠 anger: 5.39%
     ❤️ love: 3.11%
     😢 sadness: 1.98%


...


📊 Analysis Result:
   Text: I fail to the test.
   Primary Emotion: 😢 sadness (Confidence: 81.80%)
   Other possible emotions:
     😠 anger: 9.11%
     😨 fear: 5.11%
     😊 joy: 3.14%



Enter English text:  i like eat apple but it;s too expensive



📊 Analysis Result:
   Text: i like eat apple but it;s too expensive
   Primary Emotion: 😊 joy (Confidence: 94.67%)
   Other possible emotions:
     😠 anger: 2.69%
     😢 sadness: 1.46%
     ❤️ love: 0.60%



Enter English text:  I pass the important exam.



📊 Analysis Result:
   Text: I pass the important exam.
   Primary Emotion: 😊 joy (Confidence: 95.44%)
   Other possible emotions:
     😠 anger: 1.87%
     😢 sadness: 1.07%
     😨 fear: 0.68%


In [3]:
def evaluate_english_model():
    """评估英文情绪分类模型性能"""
    # 创建测试数据集
    test_samples = [
        {"text": "I'm so happy today!", "expected": "joy"},
        {"text": "This makes me really angry!", "expected": "anger"},
        {"text": "I'm scared of what might happen", "expected": "fear"},
        {"text": "I love you so much", "expected": "love"},
        {"text": "I feel so sad and lonely", "expected": "sadness"},
        {"text": "Wow! I can't believe it!", "expected": "surprise"}
    ]
    
    correct = 0
    total = len(test_samples)
    
    print("🧪 Model Performance Evaluation:")
    print("=" * 50)
    
    for sample in test_samples:
        result = predict_emotion_english(sample["text"])
        predicted = result['primary_emotion']
        expected = sample["expected"]
        
        is_correct = predicted == expected
        if is_correct:
            correct += 1
            
        status = "✅" if is_correct else "❌"
        print(f"{status} Text: '{sample['text']}'")
        print(f"   Expected: {expected}, Predicted: {predicted} ({result['primary_confidence']:.2%})")
        print()
    
    accuracy = correct / total
    print(f"📊 Overall Accuracy: {accuracy:.2%} ({correct}/{total})")
    
    return accuracy

# 运行性能评估
evaluate_english_model()

🧪 Model Performance Evaluation:
✅ Text: 'I'm so happy today!'
   Expected: joy, Predicted: joy (99.57%)

✅ Text: 'This makes me really angry!'
   Expected: anger, Predicted: anger (99.34%)

✅ Text: 'I'm scared of what might happen'
   Expected: fear, Predicted: fear (99.37%)

✅ Text: 'I love you so much'
   Expected: love, Predicted: love (79.27%)

✅ Text: 'I feel so sad and lonely'
   Expected: sadness, Predicted: sadness (99.63%)

❌ Text: 'Wow! I can't believe it!'
   Expected: surprise, Predicted: joy (75.01%)

📊 Overall Accuracy: 83.33% (5/6)


0.8333333333333334